# 9. 협업 패턴

컨텍스트 격리를 기준으로 단일 실행과 분리 실행의 최대 컨텍스트를 비교한다.

### 공통 설정

`1`과 같은 블록이다. 노트북끼리 코드를 공유하지 않는다.

In [1]:
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]

client = OpenAI()

PRICES = {MODEL: (2.50, 15.00)}
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

다섯 문서를 읽는 과제다. 문서 크기부터 본다.

In [2]:
CORPUS = Path("../../../../../data/corpus/graph")
DOCS = ["한인애국단", "윤봉길", "이봉창_의거", "김도연_(정치인)", "조선민족혁명당"]
print({d: f"{len((CORPUS / f'{d}.md').read_text(encoding='utf-8')):,}자" for d in DOCS})

{'한인애국단': '6,032자', '윤봉길': '7,774자', '이봉창_의거': '5,044자', '김도연_(정치인)': '7,974자', '조선민족혁명당': '8,958자'}


### 코어 블록

`5`에서 정의한 코어 블록에 문서 읽기와 검색 도구를 붙인다.

Part 3 에서는 에이전트가 여럿이고, 에이전트마다 모델, 지시문, 도구, 예산이 다르다. 그래서 `run`이 전역 `CONFIG`를 읽지 않고 설정을 인자 `cfg`로 받는다. 모델과 도구, `IMPL`도 설정에 넣는다.
에이전트 하나가 설정 하나이고, 워커는 자기 설정으로 도는 하네스 한 번이다. 워커를 스레드로 동시에 돌려도 전역을 바꿔 끼우지 않으므로 서로의 설정을 덮어쓰지 않는다. 이 회차부터 `12-1`까지 이 모양을 쓴다.

In [3]:
def read_doc(title):
    f = CORPUS / f"{title}.md"
    if not f.exists():
        raise FileNotFoundError(f"'{title}' 문서가 없다")
    return f.read_text(encoding="utf-8")


def grep_docs(pattern):
    return [f.stem for f in sorted(CORPUS.glob("*.md")) if pattern in f.read_text(encoding="utf-8")]

In [4]:
IMPL = {"read_doc": read_doc, "grep_docs": grep_docs}
TOOLS = [{"type": "function", "name": "read_doc", "strict": True,
          "description": "문서 한 편의 본문 전체를 돌려준다.",
          "parameters": {"type": "object",
                         "properties": {"title": {"type": "string", "description": "문서 제목"}},
                         "required": ["title"], "additionalProperties": False}},
         {"type": "function", "name": "grep_docs", "strict": True,
          "description": "낱말이 든 문서의 제목을 돌려준다.",
          "parameters": {"type": "object",
                         "properties": {"pattern": {"type": "string", "description": "찾을 낱말"}},
                         "required": ["pattern"], "additionalProperties": False}}]

CONFIG = {"model": MODEL, "instructions": "도구로 읽은 것만 근거로 삼는다.", "tools": TOOLS, "impl": IMPL,
          "max_turns": 8, "max_tokens": 200000, "max_seconds": 180, "gated": [],
          "gate_error": "error: 승인이 필요하다.", "tool_error": "error: {e}. 인자를 고쳐 다시 부른다."}

`5`의 코어에서 바뀐 것은 둘이다. `CONFIG` 대신 인자 `cfg`를 읽고, 트레이스에 턴별 입력 토큰을 하나 더 남긴다. 입력 토큰이 이 회차의 측정값이다.

In [5]:
def call_tool(name, args, cfg, approve):
    if name in cfg["gated"] and not approve:
        return cfg["gate_error"]
    try:
        return json.dumps(cfg["impl"][name](**args), ensure_ascii=False)
    except Exception as e:
        return cfg["tool_error"].format(e=e)


def run(task, cfg=CONFIG, approve=False):
    messages = [{"role": "user", "content": task}]
    trace, t0, used = [], time.perf_counter(), 0
    for turn in range(1, cfg["max_turns"] + 1):
        r = keep(client.responses.create(model=cfg["model"], instructions=cfg["instructions"],
                                         input=messages, tools=cfg["tools"], store=False))
        used += r.usage.total_tokens
        calls = [i for i in r.output if i.type == "function_call"]
        trace.append({"턴": turn, "도구": [c.name for c in calls], "입력": r.usage.input_tokens,
                      "토큰": used,
                      "초": round(time.perf_counter() - t0, 1)})
        if not calls:
            return r.output_text, "완료", trace
        if used > cfg["max_tokens"]:
            return None, "토큰 예산", trace
        if time.perf_counter() - t0 > cfg["max_seconds"]:
            return None, "시간 예산", trace
        messages += [i.model_dump() for i in r.output]
        messages += [{"type": "function_call_output", "call_id": c.call_id,
                      "output": call_tool(c.name, json.loads(c.arguments), cfg, approve)}
                     for c in calls]
    return None, "반복 예산", trace

### 한 컨텍스트에 다 넣기

한 실행에서 문서 다섯 편을 읽으므로 마지막 턴의 컨텍스트에 본문 전체가 남는다.

In [6]:
ONE = "다음 문서를 각각 읽고 한 문장으로 요약한다. " + ", ".join(DOCS)

answer, why, trace = run(ONE)
print(why, "|", answer[:200])

완료 | - **한인애국단**: 한인애국단은 김구가 1931년 상하이에서 조직한 항일무장 독립운동단체로, 이봉창 의거와 윤봉길 의거를 감행한 뒤 1936년 무렵 자연스럽게 해체된 것으로 보인다.  
- **윤봉길**: 윤봉길은 한인애국단 소속 독립운동가로 1932년 상하이 훙커우 공원에서 폭탄 의거를 일으켜 일본 고관들을 처단한 뒤 같은 해 순국하였다.  
- **


턴별 입력 토큰에서 최대값과 합계를 뽑는다.

In [7]:
solo_peak = max(t["입력"] for t in trace)
solo_total = sum(t["입력"] for t in trace)
for t in trace:
    print(f"  턴 {t['턴']}  입력 {t['입력']:7,d} 토큰  {t['도구']}")
print(f"턴 {len(trace)}  최대 컨텍스트 {solo_peak:,} 토큰  입력 합계 {solo_total:,} 토큰")

  턴 1  입력     154 토큰  ['read_doc', 'read_doc', 'read_doc', 'read_doc', 'read_doc']
  턴 2  입력  24,626 토큰  []
턴 2  최대 컨텍스트 24,626 토큰  입력 합계 24,780 토큰


### 나눠서 넣기

문서별로 실행을 분리한다. 실행마다 최대 컨텍스트와 입력 합계를 모은다.

In [8]:
peaks, totals, parts = [], [], []
for d in DOCS:
    answer, why, trace = run(f"{d} 문서를 읽고 한 문장으로 요약한다")
    peaks.append(max(t["입력"] for t in trace))
    totals.append(sum(t["입력"] for t in trace))
    parts.append(answer)
    print(f"{d:14} 최대 {peaks[-1]:7,d}  합계 {totals[-1]:7,d}")

한인애국단          최대   4,369  합계   4,492
윤봉길            최대   5,531  합계   5,652
이봉창_의거         최대   3,257  합계   3,381
김도연_(정치인)      최대   5,676  합계   5,802
조선민족혁명당        최대   6,266  합계   6,391


단일 실행과 나란히 둔다. 줄어드는 값은 입력 합계가 아니라 실행별 최대 컨텍스트다.

In [9]:
print(f"분리   최대 컨텍스트 {max(peaks):,} 토큰   입력 합계 {sum(totals):,} 토큰")
print(f"단일   최대 컨텍스트 {solo_peak:,} 토큰   입력 합계 {solo_total:,} 토큰")
print(f"\n한 번에 보는 양 {max(peaks) / solo_peak:.0%}   합계 {sum(totals) / solo_total:.0%}")

분리   최대 컨텍스트 6,266 토큰   입력 합계 25,718 토큰
단일   최대 컨텍스트 24,626 토큰   입력 합계 24,780 토큰

한 번에 보는 양 25%   합계 104%


### 격리는 총량을 줄이지 않는다

분리 실행은 지시를 반복하고 통합 단계도 필요하므로 입력 합계가 늘 수 있다. 줄어드는 값은 **실행별 최대 컨텍스트**다.

컨텍스트가 길어지면 중간 정보가 묻히거나 앞 문서의 내용이 뒤 문서 요약에 섞일 수 있다. 여기서는 품질이 아니라 자원 사용량만 측정한다.

### 나누면 안 되는 경우

앞 단계의 결과가 뒤 단계의 입력이면 실행을 독립적으로 분리할 수 없다. 1단계를 먼저 돌린다.

In [10]:
CHAIN_A = "한인애국단 문서를 읽고 그 단체를 조직한 인물의 이름만 답한다"
CHAIN_B = "위에서 찾은 인물이 등장하는 문서가 코퍼스에 몇 편인지 센다"

a, _, _ = run(CHAIN_A)
print("1단계 |", str(a)[:60])

1단계 | 김구


2단계를 따로 돌린다. 앞 단계의 결과를 모르는 채로 시작한다.

In [11]:
b, _, tb = run(CHAIN_B)
print("나눠서 2단계 |", str(b)[:90])
print("  도구", [t["도구"] for t in tb])

나눠서 2단계 | 사용자가 말한 "위에서 찾은 인물"이 현재 대화 맥락에 명시되지 않았습니다. 어느 인물을 기준으로 문서 수를 셀지 먼저 알아야 합니다.어느 인물을 말씀하시는지 현
  도구 [['grep_docs'], []]


같은 두 단계를 한 실행에 이어서 돌린다. (ReWoo와 유사)

In [12]:
joined, _, _ = run(f"{CHAIN_A}\n그다음 {CHAIN_B.replace('위에서 찾은 인물', '그 인물')}")
print("이어서    |", str(joined)[:90])

이어서    | 김구
32편


### 협업 패턴(topology) 다섯 가지

| 패턴 | 격리 대상 | 공유 대상 | 적용 조건 |
|---|---|---|---|
| 오케스트레이터-워커(orchestrator-worker) | 워커의 컨텍스트 | 오케스트레이터가 준 과제와 받은 결과 | 하위 과제가 서로 독립일 때 |
| 파이프라인(pipeline) | 각 단계의 컨텍스트 | 앞 단계의 출력 | 순서가 고정되어 있을 때 |
| 블랙보드(blackboard) | 각 에이전트의 컨텍스트 | 공용 작업공간 전체 | 작업 담당자를 미리 정할 수 없을 때 |
| 토론(debate) | 각 참가자의 근거 | 서로의 주장 | 판단이 갈리고 추가 비용을 감수할 때 |
| 계층(hierarchical) | 각 계층의 컨텍스트 | 상위 지시와 하위 요약 | 과제가 깊게 분해될 때 |

각 패턴이 한 번 도는 모습은 이렇다.

- 오케스트레이터-워커: 오케스트레이터가 과제를 하위 과제로 나눠 워커에게 하나씩 준다. 워커는 자기 몫만 보고 결과를 돌려주고, 오케스트레이터가 결과를 모아 답을 만든다. 워커끼리는 서로를 모른다. `10`이 이 패턴이다.
- 파이프라인: 에이전트를 순서대로 잇는다. 앞 에이전트의 출력이 그대로 뒤 에이전트의 입력이 된다. 초안, 검토, 최종본처럼 단계가 늘 같을 때 쓴다. 중간 단계가 틀리면 뒤 단계는 그 값을 믿고 이어 간다. 실습 12 가 이 패턴이다.
- 블랙보드: 모두가 읽고 쓰는 보드 하나를 둔다. 에이전트마다 어떤 종류의 항목을 기다릴지만 정해 두고, 보드에 그 항목이 올라오면 맡은 에이전트가 깨어나 결과를 덧붙인다. 순서를 정하는 쪽이 없고, 깨어날 에이전트가 없으면 끝난다. `12-1`에서 직접 만든다.
- 토론: 여러 에이전트가 같은 질문을 따로 푼 뒤, 상대의 풀이를 보고 자기 답을 고친다. 이것을 몇 차례 되풀이하고 마지막에 답이 같으면 합의로 본다. 호출은 참가자 수 x 라운드 수만큼 든다. `9-1`에서 다수결과 나란히 잰다.
- 계층: 오케스트레이터-워커를 여러 층으로 쌓는다. 맨 위 오케스트레이터는 중간 오케스트레이터에게 큰 덩어리를 주고, 중간 오케스트레이터가 그것을 다시 워커에게 나눈다. 위로는 요약만 올라가므로 위층의 컨텍스트가 작게 유지된다. 대신 층을 지날 때마다 요약하면서 정보가 빠진다. 이 과정에서는 따로 만들지 않는다. `10`의 워커 자리에 오케스트레이터를 다시 넣으면 된다.

### 분리 기준

하위 과제가 서로의 결과를 필요로 하면 분리하지 않는다. 공유 상태가 크면 상태 전달 비용을 계산한다. 최대 컨텍스트가 실제 병목인지 확인한다.
해당 조건이 없으면 단일 에이전트를 사용한다.

### 비용 확인

In [13]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")

호출 19회
입력 64,636 토큰   출력 1,117 토큰
누적 $0.1783
